## License

This notebook is a conversion of a Sentinel Hub evalscript and is licensed under
**CC-BY-SA-4.0**.

Original evalscript:{{URL TO THE ORIGINAL SCRIPT}}
Source: Sentinel Hub Custom Scripts (CC-BY-SA-4.0)

Conversion: Development Seed (openEO-UDP project)

# {{ALGORITHM_NAME}} with OpenEO

This notebook demonstrates {{ALGORITHM_DESCRIPTION}} using Sentinel-2 imagery with the OpenEO API.

## Overview

In this notebook, we will:
1. Connect to an OpenEO backend service
2. Define an area of interest containing {{TARGET_FEATURES}}
3. Load Sentinel-2 L2A imagery for a specific date
4. Calculate and visualize the {{ALGORITHM_NAME}} algorithm
5. Export results for further analysis

## What is {{ALGORITHM_NAME}}?

{{ALGORITHM_SCIENTIFIC_DESCRIPTION}}

### Applications

{{ALGORITHM_APPLICATIONS}}

## Environment Setup

This notebook uses the [`openeo-udp`](https://pypi.org/project/openeo-udp/) package,
which provides `ParameterManager` and ships the `{{ALGORITHM_FILENAME}}.params.py` parameter file.

- **APEx JupyterLite (in the browser):** the cell below installs `openeo-udp` from PyPI.
  The other packages this notebook needs are already available there.
- **Locally:** the cell below does nothing. Install the package beforehand, either with
  `uv sync` in a checkout of the [openeo-udp repository](https://github.com/developmentseed/openeo-udp),
  or with `pip install "openeo-udp[notebooks]"`.

In [ ]:
import sys

if sys.platform == "emscripten":  # Running in the browser (APEx JupyterLite)
    %pip install openeo-udp==0.2.0

## Import Required Libraries

We begin by importing the necessary Python libraries for data processing and visualization.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from openeo.processes import array_create, if_, and_
from openeo.api.process import Parameter

# OpenEO UDP parameter management system
from openeo_udp import ParameterManager

## Load Parameters and Connect to OpenEO Backend

Load algorithm parameters from `{{ALGORITHM_FILENAME}}.params.py`, which ships with `openeo-udp`,
and connect to an OpenEO backend with automatic endpoint selection.

To use your own parameter file, see
[Using your own parameter file](https://github.com/developmentseed/openeo-udp#using-your-own-parameter-file)
in the repository README.

In [ ]:
# Algorithm ID for UDP registration (matches the notebook file name)
_algorithm_id = "{{ALGORITHM_FILENAME}}"

# Initialize parameter manager
param_manager = ParameterManager('{{ALGORITHM_FILENAME}}.params.py')

# Display available options using the built-in helper
param_manager.print_options("{{ALGORITHM_NAME}} algorithm")

# Note: Connection will be established using the interactive widgets below
# This avoids authentication issues and ensures proper endpoint-specific handling

In [ ]:
# Connect using the first parameter set on the Copernicus Data Space endpoint
connection, current_params = param_manager.quick_connect(
    param_set="{{LOCATION_NAME}}",
    endpoint="copernicus_dataspace",
)

## Load Sentinel-2 Data

Load Sentinel-2 L2A (atmospherically corrected) data. We need specific bands for {{ALGORITHM_NAME}} calculation:

{{BAND_DESCRIPTIONS}}

In [ ]:
# Load Sentinel-2 data using the selected parameters
s2cube = connection.load_collection(
    current_params["collection"].default,
    temporal_extent=current_params["time"],
    spatial_extent=current_params["bounding_box"],
    bands=current_params["bands"].default,
    properties={
        "eo:cloud_cover": lambda x: x <= current_params["cloud_cover"],
    },
)

# Apply temporal reduction (modify as needed for your algorithm)
s2cube = s2cube.reduce_dimension(
    dimension=current_params["time_dimension"], reducer="first"
)

print("✅ Sentinel-2 data loaded successfully!")

## Implement {{ALGORITHM_NAME}} Algorithm

This function implements the {{ALGORITHM_NAME}} algorithm:

{{ALGORITHM_IMPLEMENTATION_DESCRIPTION}}

In [ ]:
def {{ALGORITHM_FUNCTION_NAME}}(data):
    """
    Apply {{ALGORITHM_NAME}} algorithm
    
    Input data array: {{INPUT_BANDS}}
    
    Returns: {{OUTPUT_DESCRIPTION}}
    """
    # Extract bands from input array
    {{BAND_EXTRACTION}}
    
    # Apply scaling if needed (for Sentinel-2 L2A data)
    {{BAND_SCALING}}
    
    # Implement algorithm calculations
    {{ALGORITHM_CALCULATIONS}}
    
    # Return result
    return {{RESULT_VARIABLE}}

## Apply Algorithm

Apply the {{ALGORITHM_NAME}} function to our Sentinel-2 data cube.

In [ ]:
# Apply algorithm on the bands dimension
result_image = s2cube.apply_dimension(dimension="bands", process={{ALGORITHM_FUNCTION_NAME}})

# Apply any necessary scaling for visualization
result_image = result_image.linear_scale_range(
    {{INPUT_MIN}}, {{INPUT_MAX}}, 0, 255
)

# Save as appropriate format for visualization
result_image_output = result_image.save_result("{{OUTPUT_FORMAT}}")

## Download and Visualize Results

Download a sample area and display the {{ALGORITHM_NAME}} results.

In [ ]:
# Define parameters for the process graph
# Base name (without extension) shared by the downloaded result and its plot
filename = f"{_algorithm_id}_{current_params['location_name'].replace(' ', '_').replace(',', '').lower()}"

# Synchronous execution (POST /result) does not accept unresolved Parameter refs,
# so materialize them with the current parameter set's defaults before download.
# The underlying parameterized graph is preserved for the UDP export cell below.
resolved = param_manager.resolve(result_image_output, current_params)
resolved.download(f"{filename}.{{FILE_EXTENSION}}")

In [ ]:
img_result = Image.open(f"{filename}.{{FILE_EXTENSION}}")
fig = plt.figure(figsize=(12, 10))
plt.imshow(img_result)
plt.title(f"{{ALGORITHM_NAME}}\n{current_params['location_name']}", fontsize=16)
plt.axis("off")

# Preview image for the APEx Algorithm Catalogue record (used by maintainers when
# registering the algorithm, see CONTRIBUTING.md)
fig.savefig(f"{filename}_preview.png", bbox_inches="tight", dpi=150)
plt.show()

## Interpretation Guide

### {{ALGORITHM_NAME}} Results:

{{RESULTS_INTERPRETATION}}

### Applications:

{{DETAILED_APPLICATIONS}}

### Limitations:

{{ALGORITHM_LIMITATIONS}}

In [ ]:
# Export the process graph for reuse
import json
from pathlib import Path

process_graph = {
    "process_graph": result_image_output.flat_graph(),
    "parameters": [
        current_params["time"].to_dict(),
        current_params["bounding_box"].to_dict(),
        current_params["cloud_cover"].to_dict()
    ],
    "id": _algorithm_id,
    "summary": "{{ALGORITHM_SUMMARY}}",
    "description": (
       "{{ALGORITHM_DESCRIPTION}}"
    )
}

# Save the process graph
udp_path = Path(f"{_algorithm_id}.json")
with open(udp_path, "w") as f:
    json.dump(process_graph, f, indent=2)

print(f"✅ Process graph saved to {udp_path}")

In [ ]:
# Note: Pick the keywords from this list
# Remove this list after
accepted_keywords = [
    {
        "type": "string",
        "enum": [
            "Agriculture",
            "Land Use/Land Cover Classification",
            "Vegetation",
            "Normalized Difference Vegetation Index (NDVI)",
            "Leaf Area Index (LAI)",
            "Fraction of Absorbed Photosynthetic Active Radiation",
            "(fAPAR)",
            "Fraction of Vegetation Coverage (fCOVER)",
            "Normalized Difference Water Index (NDWI)",
            "Brightness",
            "Energy Production/Use",
            "Wind Energy Production/Use",
            "Solar Energy Production/Use",
            "Natural Hazards",
            "Wildfires",
            "Sentinel-1",
            "Sentinel-2",
            "Sentinel-3",
            "Landsat-8",
            "Digital Elevation/Terrain Model (DEM)",
            "ECMWF ERA5",
            "Data Analysis and Visualization",
            "Statistical Applications"
        ]
    }
]

# Export necessary metadata to register the process graph in APEx Algorithm Catalogue 
if "_algorithm_id" not in globals():
    _algorithm_id = "{{ALGORITHM_FILENAME}}"

# Path of this notebook in the openeo-udp repository
_nb_href = "notebooks/{{NOTEBOOK_FOLDER}}/{{ALGORITHM_FILENAME}}.ipynb"

# Notebook metadata
metadata = {
    "id" : _algorithm_id,
    "title" : "{{ALGORITHM_TITLE}}",
    "description" : "{{PROCESS_GRAPH_DESCRIPTION}}",
    "keywords" : "{{KEYWORDS_LIST}}",
    "themes" : "{{THEMES_LIST}}",
    "created" : "{{DATE_OF_ALGORITHM_CREATION}}",
    "updated" : "{{DATE_OF_ALGORITM_LAST_UPDATE}}",
    "license" : "CC-BY-SA-4.0",
    "openeo_backend_title" : "{{TARGET_OPENEO_BACKEND}}",
    "openeo_backend_url" : "{{TARGET_OPENEO_BACKEND_URL}}",
    "notebook_github_location" : _nb_href,
    "collection_id" : "{{PRODUCT_COLLECTION_ID}}",
    "attribution" : {
        "original_script" : "{{URL_TO_ORIGINAL_EVALSCRIPT}}",
        "authors" : {{["LIST", "OF", "EVALSCRIPT", "AUTHORS"]}},
        "source_repository" : "https://github.com/sentinel-hub/custom-scripts",
        "citation" : None
    }
}

## References and Attribution

### Original Script:
{{ORIGINAL_SCRIPT_ATTRIBUTION}}

### Scientific References:
{{SCIENTIFIC_REFERENCES}}

### OpenEO Conversion:
- **Conversion Date**: {{CONVERSION_DATE}}
- **OpenEO Framework**: Adapted for openEO API and process graph structure
- **Backend Tested**: {{BACKEND_TESTED}}

## Conclusion

This notebook successfully demonstrates the conversion of the {{ALGORITHM_NAME}} algorithm to an OpenEO User-Defined Process. The implementation:

✅ **Maintains Scientific Accuracy**: Preserves the original algorithm's methodology

✅ **Provides Flexible Parameter Management**: Works with multiple locations and backends

✅ **Follows OpenEO Standards**: Uses parameterized process graphs for reusability

✅ **Includes Comprehensive Documentation**: Provides interpretation guides and usage examples

### Key Achievements:

1. **Algorithm Implementation**: Successfully implemented {{ALGORITHM_NAME}} using OpenEO processes
2. **Parameter Management**: Integrated with the OpenEO UDP parameter system
3. **Multi-backend Support**: Works across different OpenEO endpoints
4. **Process Graph Export**: Generated reusable UDP definition

This conversion makes the {{ALGORITHM_NAME}} algorithm accessible to the broader OpenEO ecosystem while maintaining its scientific rigor and practical utility.